In [1]:
!pip install easyocr sentence-transformers supabase python-dotenv pdf2image pypdf
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [3]:
import os
import time
import shutil
import re
import torch
import numpy as np
import easyocr
from pathlib import Path
from pypdf import PdfReader
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from supabase import create_client
from dotenv import load_dotenv
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Setup Paths - UPDATE THESE to match your actual Drive folders
DRIVE_BASE = Path("/content/drive/MyDrive/AILEVATED")
DOTENV_PATH = DRIVE_BASE / ".env"
NEW_PDFS_DIR = DRIVE_BASE / "pdfs" / "new"
PROCESSED_PDFS_DIR = DRIVE_BASE / "pdfs" / "processed"

# Load Environment Variables
if DOTENV_PATH.exists():
    load_dotenv(DOTENV_PATH)
else:
    print(f"❌ Error: .env not found at {DOTENV_PATH}")

URL = os.getenv("SUPABASE_URL")
KEY = os.getenv("SUPABASE_KEY")

if not URL or not KEY:
    raise ValueError("Supabase credentials missing. Check your .env file path.")

supabase = create_client(URL, KEY)

# Ensure folders exist
NEW_PDFS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PDFS_DIR.mkdir(parents=True, exist_ok=True)

# 3. Initialize Models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {device.upper()}")

print("🧠 Loading Multilingual Embedding Model...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# ── Extraction Engine ────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """Cleans both Arabic artifacts and English extra whitespace."""
    text = re.sub(r'\(cid:\d+\)', '', text)
    text = text.replace('ﺩ', ' ') 
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def detect_language(text: str, filename: str) -> str:
    """Detects if content is Arabic or English."""
    if "english" in filename.lower() or "en" in filename.lower():
        return "en"
    return "ar" if re.search(r'[\u0600-\u06FF]', text) else "en"

def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200):
    """Simple chunking for processing."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
    return chunks

def extract_text_via_ocr(pdf_path: Path):
    print(f"📸 Starting OCR for: {pdf_path.name}")
    try:
        # EasyOCR auto-detects GPU in Colab
        reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
        
        # In Colab/Linux, poppler is system-wide, no path needed
        pages = convert_from_path(str(pdf_path), dpi=300)
        
        full_text = ""
        for i, page in enumerate(pages):
            print(f"📄 Reading Page {i+1}/{len(pages)}...")
            img = np.array(page)
            results = reader.readtext(img, detail=0)
            full_text += " ".join(results) + "\n"
        
        return clean_text(full_text)
    except Exception as e:
        print(f"⚠️ OCR Failed for {pdf_path.name}: {e}")
        return ""

# ── Database Logic ───────────────────────────────────────────────────────

def is_already_processed(filename: str):
    """Duplicate prevention."""
    result = supabase.table("curriculum_chunks").select("id").eq("source", filename).limit(1).execute()
    return len(result.data) > 0

def store_chunks(chunks: list, source: str, lang: str):
    batch_size = 25 
    print(f"📤 Uploading {len(chunks)} chunks to Supabase...")
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        data_to_insert = []
        for chunk in batch:
            embedding = embed_model.encode(chunk).tolist()
            data_to_insert.append({
                "content": chunk,
                "embedding": embedding,
                "source": source,
                "language": lang,
                "grade": "all",
                "subject": "general"
            })
        try:
            supabase.table("curriculum_chunks").insert(data_to_insert).execute()
        except Exception as e:
            print(f"❌ Batch upload error: {e}")
            time.sleep(2)

# ── Main Loop ────────────────────────────────────────────────────────────

def run_ingestion():
    files = list(NEW_PDFS_DIR.glob("*.pdf"))
    if not files:
        print(f"📭 No new PDFs found in {NEW_PDFS_DIR}")
        return

    for pdf_file in files:
        print(f"\n🔎 Processing: {pdf_file.name}")
        
        if is_already_processed(pdf_file.name):
            print(f"⏩ Skipping {pdf_file.name} (Already in database)")
            shutil.move(str(pdf_file), str(PROCESSED_PDFS_DIR / pdf_file.name))
            continue

        start_time = time.time()
        
        # 1. OCR
        raw_text = extract_text_via_ocr(pdf_file)
        if not raw_text:
            continue
            
        # 2. Meta
        lang = detect_language(raw_text, pdf_file.name)
        
        # 3. Chunk
        chunks = chunk_text(raw_text)
        
        # 4. Upload
        store_chunks(chunks, pdf_file.name, lang)
        
        # 5. Move
        try:
            shutil.move(str(pdf_file), str(PROCESSED_PDFS_DIR / pdf_file.name))
            print(f"✅ Success! Moved to processed folder.")
        except Exception as e:
            print(f"⚠️ Error moving file: {e}")
        
        duration = time.time() - start_time
        print(f"⏱️ Total time for {pdf_file.name}: {duration:.1f}s")

if __name__ == "__main__":
    run_ingestion()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Running on: CUDA
🧠 Loading Multilingual Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📭 No new PDFs found in /content/drive/MyDrive/AILEVATED/pdfs/new


In [4]:
import os
import time
import shutil
import re
import torch
import numpy as np
import easyocr
from pathlib import Path
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from supabase import create_client
from dotenv import load_dotenv
from google.colab import drive

# ── Mount Drive ───────────────────────────────────────────────────────────
drive.mount('/content/drive')

DRIVE_BASE = Path("/content/drive/MyDrive/AILEVATED")
DOTENV_PATH = DRIVE_BASE / ".env"
NEW_PDFS_DIR = DRIVE_BASE / "pdfs" / "new"
PROCESSED_PDFS_DIR = DRIVE_BASE / "pdfs" / "processed"

load_dotenv(DOTENV_PATH)

supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {device.upper()}")

embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# ── Metadata from folder structure ────────────────────────────────────────
# Expected: pdfs/new/{grade}/{track}/{subject}/file.pdf
def extract_metadata_from_path(pdf_path: Path) -> dict:
    parts = pdf_path.parts
    try:
        new_idx = parts.index('new')
    except ValueError:
        return {"grade": "unknown", "track": "unknown", "subject": "general"}
    
    remaining = parts[new_idx + 1:]
    
    metadata = {"grade": "unknown", "track": "unknown", "subject": "general"}
    
    if len(remaining) >= 1: metadata["grade"] = remaining[0]
    if len(remaining) >= 2: metadata["track"] = remaining[1]
    if len(remaining) >= 3 and not remaining[2].endswith('.pdf'):
        metadata["subject"] = remaining[2]
    
    return metadata


# ── Text & Language ───────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = re.sub(r'\(cid:\d+\)', '', text)
    text = text.replace('ﺩ', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def detect_language(text: str, subject: str) -> str:
    if subject.lower() in ["english", "anglais"]: return "en"
    if subject.lower() in ["french", "français", "fr", "francais"]: return "fr"
    if re.search(r'[\u0600-\u06FF]', text): return "ar"
    return "fr"


def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list:
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
    return chunks


def extract_text_via_ocr(pdf_path: Path) -> str:
    print(f"📸 OCR: {pdf_path.name}")
    try:
        reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
        pages = convert_from_path(str(pdf_path), dpi=300)
        full_text = ""
        for i, page in enumerate(pages):
            print(f"  📄 Page {i+1}/{len(pages)}")
            img = np.array(page)
            results = reader.readtext(img, detail=0)
            full_text += " ".join(results) + "\n"
        return clean_text(full_text)
    except Exception as e:
        print(f"⚠️ OCR failed: {e}")
        return ""


# ── Database ──────────────────────────────────────────────────────────────
def is_already_processed(filename: str) -> bool:
    result = supabase.table("curriculum_chunks").select("id").eq("source", filename).limit(1).execute()
    return len(result.data) > 0


def store_chunks(chunks, source, language, grade, track, subject):
    batch_size = 25
    print(f"📤 Uploading {len(chunks)} chunks | grade={grade} track={track} subject={subject}")
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        data = []
        for chunk in batch:
            embedding = embed_model.encode(chunk).tolist()
            data.append({
                "content": chunk,
                "embedding": embedding,
                "source": source,
                "language": language,
                "grade": grade,
                "track": track,
                "subject": subject
            })
        try:
            supabase.table("curriculum_chunks").insert(data).execute()
            print(f"  ✅ Batch {i//batch_size + 1} uploaded")
        except Exception as e:
            print(f"  ❌ Batch failed: {e}")
            time.sleep(2)
        time.sleep(0.3)


# ── Main ──────────────────────────────────────────────────────────────────
def run_ingestion():
    # Find ALL pdfs recursively inside new/
    pdf_files = list(NEW_PDFS_DIR.rglob("*.pdf"))
    
    if not pdf_files:
        print(f"📭 No PDFs found in {NEW_PDFS_DIR}")
        return
    
    print(f"Found {len(pdf_files)} PDFs\n")
    
    for pdf_path in pdf_files:
        print(f"\n{'='*50}")
        print(f"📄 {pdf_path.name}")
        
        if is_already_processed(pdf_path.name):
            print(f"⏩ Already in database — skipping")
            continue
        
        meta = extract_metadata_from_path(pdf_path)
        print(f"📁 grade={meta['grade']} | track={meta['track']} | subject={meta['subject']}")
        
        start = time.time()
        
        text = extract_text_via_ocr(pdf_path)
        if not text:
            print("❌ No text — skipping")
            continue
        
        lang = detect_language(text, meta['subject'])
        print(f"🌐 Language: {lang}")
        
        chunks = chunk_text(text)
        print(f"📦 {len(chunks)} chunks")
        
        store_chunks(
            chunks=chunks,
            source=pdf_path.name,
            language=lang,
            grade=meta['grade'],
            track=meta['track'],
            subject=meta['subject']
        )
        
        # Move to processed preserving structure
        processed_path = PROCESSED_PDFS_DIR / Path(*pdf_path.parts[pdf_path.parts.index('new')+1:])
        processed_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(pdf_path), str(processed_path))
        
        duration = time.time() - start
        print(f"✅ Done in {duration:.1f}s")


run_ingestion()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Running on: CUDA


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 66 PDFs


📄 educational-program-74.pdf


📁 grade=2bac | track=economie | subject=arabic
📸 OCR: educational-program-74.pdf
Progress: |██████████████████████████████████████████████████| 100.0% Complete  📄 Page 1/4
  📄 Page 2/4
  📄 Page 3/4
  📄 Page 4/4
🌐 Language: ar
📦 2 chunks
📤 Uploading 2 chunks | grade=2bac track=economie subject=arabic
  ✅ Batch 1 uploaded
✅ Done in 20.1s

📄 le-programme-pedagogique-comptabilite-2-bac-sciences-economiques.pdf
📁 grade=2bac | track=economie | subject=comptabilite
📸 OCR: le-programme-pedagogique-comptabilite-2-bac-sciences-economiques.pdf
  📄 Page 1/3
  📄 Page 2/3
  📄 Page 3/3
🌐 Language: ar
📦 4 chunks
📤 Uploading 4 chunks | grade=2bac track=economie subject=comptabilite
  ✅ Batch 1 uploaded
✅ Done in 17.7s

📄 national-review-framework-11.pdf
📁 grade=2bac | track=economie | subject=comptabilite
📸 OCR: national-review-framework-11.pdf
  📄 Page 1/17
  📄 Page 2/17
  📄 Page 3/17
  📄 Page 4/17
  📄 Page 5/17
  📄 Page 6/17
  📄 Page 7/17
  📄 Page 8/17
  📄 Page 9/17
  📄 Page 10/17
  📄 Page 11/17
  📄 